In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
%pip install -q xgboost optuna joblib pyarrow

# 02 — Training Replicas

For each window pair (A, B) this notebook:

1. Tunes hyperparameters **per window** using Optuna with TimeSeriesSplit, optimising PR-AUC.
2. Trains **R replicas** per window using stratified bootstrap sampling + different random seeds.
3. Evaluates on the common evaluation slice E_{A,B} (replica-averaged predictions, PR-AUC and ROC-AUC).
4. Identifies the **flagged set** F_{A,B} as the top K_FRAC fraction of eval instances ranked by max(p_hat_A, p_hat_B).

**Model types supported:** `'xgboost'` | `'logreg'`  ← set via `MODEL_TYPE` in the config cell.
MLP-PLR is in notebook 02b.

**Input:** `data/processed/`, `data/windows/window_config.json`
**Output per pair:** `data/models/{model_type}/pair_{pid:02d}/`
- `replicas_A/model_r{r}.joblib`, `replicas_B/model_r{r}.joblib` — R fitted models
- `replicas_A/seeds_r{r}.json`, `replicas_B/seeds_r{r}.json` — exact bootstrap/model seeds
- `hparams_A.json`, `hparams_B.json` — tuned hyperparameters
- `reference_scaler.joblib` — StandardScaler fit on window A's numeric features (used by notebook 04 for covariate drift; NOT used by either model internally)
- `predictions.npz` — see schema below
- `coef_A.npy`, `coef_B.npy` *(LR only)* — coefficient tensors, shape `(R, p)`

For XGBoost runs only:
- `replicas_A/training_log_r{r}.csv`, `replicas_B/training_log_r{r}.csv` — per-replica eval-metric curve

**`predictions.npz` schema (unchanged across model types):**

| Key | Shape | Meaning |
|---|---|---|
| `preds_A` | `(R, n_eval)` | per-replica positive-class probability, window A |
| `preds_B` | `(R, n_eval)` | per-replica positive-class probability, window B |
| `p_hat_A` | `(n_eval,)` | replica-averaged probability, window A |
| `p_hat_B` | `(n_eval,)` | replica-averaged probability, window B |
| `flagged_idx` | `(n_flagged,)` | local positions within `idx_eval` of the flagged set |
| `Y_eval` | `(n_eval,)` | true labels |
| `pr_auc_A` | scalar | average precision of `p_hat_A` |
| `pr_auc_B` | scalar | average precision of `p_hat_B` |
| `roc_auc_A` | scalar | ROC-AUC of `p_hat_A` |
| `roc_auc_B` | scalar | ROC-AUC of `p_hat_B` |
| `per_replica_pr_auc_A` | `(R,)` | PR-AUC of each individual replica, window A |
| `per_replica_pr_auc_B` | `(R,)` | PR-AUC of each individual replica, window B |

**Replica design:** Each replica differs by (1) a different random seed and (2) a stratified bootstrap sample.


In [13]:
import json
import joblib
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score

import xgboost as xgb
import sklearn as _sklearn

# Silence the benign XGBoost warning that fires once per booster build:
#   "Parameters: { "enable_categorical" } are not used."
# It comes from passing classifier-level kwargs through xgb.train(...). The
# behaviour is correct — XGBoost just doesn't consume the flag in that path.
warnings.filterwarnings(
    'ignore',
    message=r'.*Parameters:.*enable_categorical.*are not used.*',
)

# Silence Optuna chatter — we print our own summaries.
optuna.logging.set_verbosity(optuna.logging.WARNING)

WORKSPACE = Path('/content/drive/MyDrive/Homesite_workspace')
PROC_DIR  = WORKSPACE / 'data' / 'processed'
WIN_DIR   = WORKSPACE / 'data' / 'windows'

# ── Fixed XGBoost training knobs (not tuned) ──────────────────────────────
# enable_categorical is intentionally NOT set here: the feature matrix is
# fully numeric (preprocessing in nb 00 ordinal-encodes everything), and the
# flag is silently ignored by xgb.train() while triggering a noisy warning.
XGB_FIXED = dict(
    objective        = 'binary:logistic',
    eval_metric      = 'aucpr',
    tree_method      = 'hist',
    n_jobs           = -1,
)

# ── Fixed LR training knobs (not tuned). Solver picked for sklearn compatibility ──
_sk_ver = tuple(int(x) for x in _sklearn.__version__.split('.')[:2])
_lr_solver = 'newton-cholesky' if _sk_ver >= (1, 2) else 'lbfgs'

LR_FIXED = dict(
    solver   = _lr_solver,
    max_iter = 500,
)

# ── Tuning configuration ──────────────────────────────────────────────────
N_TRIALS       = 50    # Optuna trials per window (XGBoost)
N_TRIALS_LR    = 15    # Optuna trials per window (Logistic Regression)
CV_N_SPLITS    = 3     # TimeSeriesSplit folds inside each window
ES_ROUNDS      = 50    # early stopping rounds (XGBoost only)
MAX_BOOST_RND  = 2000  # hard cap on n_estimators (XGBoost only)
VAL_TAIL_FRAC  = 0.15  # fraction of bootstrap used as ES validation set (XGBoost only)

print('Imports OK')
print(f'LR solver: {_lr_solver}  (sklearn {_sklearn.__version__})')
print(f'XGBoost  : {xgb.__version__}')

Imports OK
LR solver: newton-cholesky  (sklearn 1.6.1)
XGBoost  : 3.2.0


In [14]:
# ── Model type — the ONE knob that switches between branches ──────────────
MODEL_TYPE = 'logreg'   # 'xgboost' or 'logreg'

assert MODEL_TYPE in ('xgboost', 'logreg'), f'Unknown MODEL_TYPE: {MODEL_TYPE}'

MODEL_DIR = WORKSPACE / 'data' / 'models' / MODEL_TYPE
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODEL_TYPE : {MODEL_TYPE}')
print(f'MODEL_DIR  : {MODEL_DIR}')

MODEL_TYPE : logreg
MODEL_DIR  : /content/drive/MyDrive/Homesite_workspace/data/models/logreg


## 1. Load processed data and window config

In [15]:
X_df = pd.read_parquet(PROC_DIR / 'X.parquet')
X    = X_df.values.astype(np.float32)
Y    = np.load(PROC_DIR / 'Y.npy').astype(np.int8)

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names_json = json.load(f)

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R           = config['parameters']['R']
K_FRAC      = config['parameters']['K_FRAC']
PAIR_STRIDE = config['parameters']['PAIR_STRIDE']
TIME_UNIT   = config['parameters']['TIME_UNIT']
pairs       = config['pairs']

# Column-index lookups for the reference scaler (numeric cols only).
all_cols     = feature_names_json['all']
num_col_idx  = [all_cols.index(c) for c in feature_names_json['num']]
bin_col_idx  = [all_cols.index(c) for c in feature_names_json['bin']]

print(f'X: {X.shape}, Y: {Y.shape}')
print(f'R={R}, K_FRAC={K_FRAC}, PAIR_STRIDE={PAIR_STRIDE}, {len(pairs)} pairs')
print(f'Numeric: {len(num_col_idx)}   Binary: {len(bin_col_idx)}   Total: {X.shape[1]}')

X: (260753, 317), Y: (260753,)
R=3, K_FRAC=0.1, PAIR_STRIDE=4, 5 pairs
Numeric: 280   Binary: 37   Total: 317


## 2. Helper functions

The same building blocks for both model types — bootstrap sampling, top-K flagging — plus per-model-type tuning and training functions.

In [16]:
SEED_BASE = 42

def stratified_bootstrap(idx: np.ndarray, Y: np.ndarray, seed: int) -> np.ndarray:
    """Sample with replacement, preserving class balance within `idx`."""
    rng = np.random.default_rng(seed)
    Y_sub = Y[idx]
    pos = idx[Y_sub == 1]
    neg = idx[Y_sub == 0]
    boot_pos = rng.choice(pos, size=len(pos), replace=True)
    boot_neg = rng.choice(neg, size=len(neg), replace=True)
    return np.concatenate([boot_pos, boot_neg])


def compute_flagged_topk(p_hat_A: np.ndarray, p_hat_B: np.ndarray, k_frac: float) -> np.ndarray:
    """Top-k fraction of eval rows ranked by max(p_hat_A, p_hat_B)."""
    score = np.maximum(p_hat_A, p_hat_B)
    n_flag = max(int(np.ceil(len(score) * k_frac)), 1)
    return np.argsort(-score)[:n_flag].astype(np.int64)


def predict_proba_pos(model, X_eval: np.ndarray) -> np.ndarray:
    """Return positive-class probability for either an XGB Booster or sklearn pipeline."""
    if isinstance(model, xgb.Booster):
        return model.predict(xgb.DMatrix(X_eval))
    return model.predict_proba(X_eval)[:, 1]

In [17]:
# ═════════════════════════════════════════════════════════════════════════════
# XGBoost — tuning + replica training with chronological ES tail
# ═════════════════════════════════════════════════════════════════════════════

def tune_hyperparameters_xgboost(window_idx: np.ndarray,
                                 X_all: np.ndarray,
                                 Y_all: np.ndarray,
                                 study_seed: int,
                                 n_trials: int = N_TRIALS) -> dict:
    """Optuna search inside the window using TimeSeriesSplit. Optimises PR-AUC.

    Inputs are step-sorted (notebook 01 returns indices in chronological order),
    so TimeSeriesSplit gives proper forward-only folds.
    """
    X_win = X_all[window_idx]
    Y_win = Y_all[window_idx]
    tscv  = TimeSeriesSplit(n_splits=CV_N_SPLITS)

    def objective(trial: optuna.Trial) -> float:
        params = {
            **XGB_FIXED,
            'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'max_depth':        trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_float('min_child_weight', 1e-2, 10.0, log=True),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'reg_alpha':        trial.suggest_float('reg_alpha',  1e-3, 10.0, log=True),
        }

        pr_aucs = []
        for fold, (tr, va) in enumerate(tscv.split(X_win)):
            dtrain = xgb.DMatrix(X_win[tr], label=Y_win[tr])
            dval   = xgb.DMatrix(X_win[va], label=Y_win[va])
            booster = xgb.train(
                {**params, 'seed': study_seed + fold},
                dtrain,
                num_boost_round=MAX_BOOST_RND,
                evals=[(dval, 'val')],
                early_stopping_rounds=ES_ROUNDS,
                verbose_eval=False,
            )
            p = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))
            pr_aucs.append(average_precision_score(Y_win[va], p))

        return float(np.mean(pr_aucs))

    sampler = TPESampler(seed=study_seed)
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return dict(study.best_trial.params)


def train_replica_with_es(X_train: np.ndarray, Y_train: np.ndarray,
                          X_val: np.ndarray, Y_val: np.ndarray,
                          params: dict, model_seed: int):
    """Train one XGBoost replica on a bootstrap with chronological-tail early stopping.

    X_val/Y_val is the chronological tail of the original window, extracted by
    the caller BEFORE bootstrapping. This way ES monitors the most-recent slice
    of data rather than a random tail of a shuffled bootstrap.
    """
    dtrain = xgb.DMatrix(X_train, label=Y_train)
    dval   = xgb.DMatrix(X_val,   label=Y_val)

    evals_result = {}
    booster = xgb.train(
        {**XGB_FIXED, **params, 'seed': model_seed},
        dtrain,
        num_boost_round=MAX_BOOST_RND,
        evals=[(dval, 'val')],
        early_stopping_rounds=ES_ROUNDS,
        verbose_eval=False,
        evals_result=evals_result,
    )
    log = pd.DataFrame({
        'iter':    range(len(evals_result['val']['aucpr'])),
        'val_pr':  evals_result['val']['aucpr'],
    })
    return booster, log

In [18]:
# ═════════════════════════════════════════════════════════════════════════════
# Logistic Regression — tuning + replica training (full bootstrap, no ES)
# ═════════════════════════════════════════════════════════════════════════════

def tune_hyperparameters_logreg(window_idx: np.ndarray,
                                X_all: np.ndarray,
                                Y_all: np.ndarray,
                                study_seed: int,
                                n_trials: int = N_TRIALS_LR) -> dict:
    """Optuna search for LR — tunes only C with a fixed l2 penalty."""
    X_win = X_all[window_idx]
    Y_win = Y_all[window_idx]
    tscv  = TimeSeriesSplit(n_splits=CV_N_SPLITS)

    def objective(trial: optuna.Trial) -> float:
        params = {
            'C':       trial.suggest_float('C', 1e-3, 10.0, log=True),
            'penalty': 'l2',
        }
        pr_aucs = []
        for fold, (tr, va) in enumerate(tscv.split(X_win)):
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('model',  LogisticRegression(**{**LR_FIXED, **params,
                                                 'random_state': study_seed + fold})),
            ])
            pipe.fit(X_win[tr], Y_win[tr])
            p = pipe.predict_proba(X_win[va])[:, 1]
            pr_aucs.append(average_precision_score(Y_win[va], p))
        return float(np.mean(pr_aucs))

    sampler = TPESampler(seed=study_seed)
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return dict(study.best_trial.params)


def train_replica_logreg(X_train: np.ndarray, Y_train: np.ndarray,
                         params: dict, model_seed: int) -> Pipeline:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(**{**LR_FIXED, **params,
                                         'random_state': model_seed})),
    ])
    pipe.fit(X_train, Y_train)
    return pipe

## 3. Main training loop

Per pair: tune → bootstrap → train R replicas → evaluate → save. Skips pairs whose `predictions.npz` already exists (resume-friendly).

In [19]:
performance_log = []

for p in pairs:
    pid       = p['pair_id']
    pair_dir  = MODEL_DIR / f'pair_{pid:02d}'
    pred_file = pair_dir / 'predictions.npz'

    # ── Resume: skip pairs that already finished with a complete schema ──
    if pred_file.exists():
        data = np.load(pred_file)
        if all(k in data.files for k in ('roc_auc_A', 'roc_auc_B')):
            print(f'Pair {pid:02d}: already done, skipping.')
            performance_log.append({
                'pair_id':   pid,
                'pr_auc_A':  float(data['pr_auc_A']),
                'pr_auc_B':  float(data['pr_auc_B']),
                'roc_auc_A': float(data['roc_auc_A']),
                'roc_auc_B': float(data['roc_auc_B']),
                'n_flagged': int(data['flagged_idx'].shape[0]),
            })
            continue
        print(f'Pair {pid:02d}: stale predictions.npz — re-running.')

    print(
        f'\n── Pair {pid:02d}: '
        f'A_end={p["step_label_A"]}  '
        f'B_end={p["step_label_B"]}  '
        f'eval={p["eval_start_label"]}→{p["eval_end_label"]}  '
        f'|A|={p["n_train_A"]:,} |B|={p["n_train_B"]:,} |eval|={p["n_eval"]:,}  '
        f'PAIR_STRIDE={PAIR_STRIDE} ──'
    )

    idx_A    = np.array(p['idx_A'],    dtype=np.int64)
    idx_B    = np.array(p['idx_B'],    dtype=np.int64)
    idx_eval = np.array(p['idx_eval'], dtype=np.int64)

    assert len(set(idx_A.tolist()) & set(idx_eval.tolist())) == 0, 'idx_A and idx_eval overlap!'
    assert len(set(idx_B.tolist()) & set(idx_eval.tolist())) == 0, 'idx_B and idx_eval overlap!'

    X_eval = X[idx_eval]
    Y_eval = Y[idx_eval]

    pair_dir.mkdir(parents=True, exist_ok=True)
    dir_A = pair_dir / 'replicas_A'
    dir_B = pair_dir / 'replicas_B'
    dir_A.mkdir(exist_ok=True)
    dir_B.mkdir(exist_ok=True)

    # ── Reference scaler (used by notebook 04 for covariate-drift; not used by either model) ──
    ref_scaler = StandardScaler()
    ref_scaler.fit(X[idx_A][:, num_col_idx])
    joblib.dump(ref_scaler, pair_dir / 'reference_scaler.joblib')

    # ── Tune hyperparameters per window ──
    print('  Tuning window A ...')
    if MODEL_TYPE == 'xgboost':
        hparams_A = tune_hyperparameters_xgboost(idx_A, X, Y, study_seed=SEED_BASE + pid * 10 + 1)
    else:
        hparams_A = tune_hyperparameters_logreg(idx_A, X, Y, study_seed=SEED_BASE + pid * 10 + 1)
    with open(pair_dir / 'hparams_A.json', 'w') as f:
        json.dump(hparams_A, f, indent=2)
    print(f'    best A params: {hparams_A}')

    print('  Tuning window B ...')
    if MODEL_TYPE == 'xgboost':
        hparams_B = tune_hyperparameters_xgboost(idx_B, X, Y, study_seed=SEED_BASE + pid * 10 + 2)
    else:
        hparams_B = tune_hyperparameters_logreg(idx_B, X, Y, study_seed=SEED_BASE + pid * 10 + 2)
    with open(pair_dir / 'hparams_B.json', 'w') as f:
        json.dump(hparams_B, f, indent=2)
    print(f'    best B params: {hparams_B}')

    # ── Chronological ES tail (XGBoost only). LR uses the full bootstrap. ──
    if MODEL_TYPE == 'xgboost':
        n_es_A     = max(int(np.ceil(len(idx_A) * VAL_TAIL_FRAC)), 1)
        idx_A_boot = idx_A[:-n_es_A]
        X_va_A     = X[idx_A[-n_es_A:]]
        Y_va_A     = Y[idx_A[-n_es_A:]]
        n_es_B     = max(int(np.ceil(len(idx_B) * VAL_TAIL_FRAC)), 1)
        idx_B_boot = idx_B[:-n_es_B]
        X_va_B     = X[idx_B[-n_es_B:]]
        Y_va_B     = Y[idx_B[-n_es_B:]]
    else:
        idx_A_boot = idx_A
        idx_B_boot = idx_B

    # ── Train R replicas for window A ──
    preds_A = np.zeros((R, len(idx_eval)), dtype=np.float32)
    per_rep_pr_auc_A = np.zeros(R, dtype=np.float32)
    if MODEL_TYPE == 'logreg':
        coef_A = np.zeros((R, X.shape[1]), dtype=np.float32)

    for r in range(R):
        boot_seed  = SEED_BASE + pid * 10_000 + r * 2          # A-bootstrap seeds: even
        model_seed = SEED_BASE + pid * 10_000 + r * 2 + 1      # A-model seeds:    odd
        boot_idx   = stratified_bootstrap(idx_A_boot, Y, seed=boot_seed)
        X_tr = X[boot_idx]
        Y_tr = Y[boot_idx]

        if MODEL_TYPE == 'xgboost':
            model, log = train_replica_with_es(X_tr, Y_tr, X_va_A, Y_va_A, hparams_A, model_seed)
            preds_A[r] = predict_proba_pos(model, X_eval)
            log.to_csv(dir_A / f'training_log_r{r}.csv', index=False)
        else:
            model = train_replica_logreg(X_tr, Y_tr, hparams_A, model_seed)
            preds_A[r] = model.predict_proba(X_eval)[:, 1]
            coef_A[r]  = model.named_steps['model'].coef_.ravel()

        joblib.dump(model, dir_A / f'model_r{r}.joblib', compress=3)
        per_rep_pr_auc_A[r] = average_precision_score(Y_eval, preds_A[r])
        with open(dir_A / f'seeds_r{r}.json', 'w') as f:
            json.dump({'bootstrap_seed': int(boot_seed),
                       'model_seed':     int(model_seed)}, f, indent=2)
        print(f'  A replica {r}: PR-AUC = {per_rep_pr_auc_A[r]:.4f}')

    if MODEL_TYPE == 'logreg':
        np.save(pair_dir / 'coef_A.npy', coef_A)
        print(f'  Saved coef_A.npy  shape={coef_A.shape}')

        # Full-window LR coefficient: one deterministic fit on all of window A,
        # no bootstrap, no replica averaging. Used by notebook 04 to render the
        # transparent A-vs-B coefficient trajectory plot. Coefficients are in
        # the per-window standardized basis (scaler fit on window A only),
        # exactly matching how each replica is trained.
        full_pipe_A = Pipeline([
            ('scaler', StandardScaler()),
            ('model',  LogisticRegression(**{**LR_FIXED, **hparams_A,
                                             'random_state': SEED_BASE + pid * 10})),
        ])
        full_pipe_A.fit(X[idx_A], Y[idx_A])
        coef_A_full = full_pipe_A.named_steps['model'].coef_.ravel().astype(np.float32)
        np.save(pair_dir / 'coef_A_full.npy', coef_A_full)
        print(f'  Saved coef_A_full.npy  shape={coef_A_full.shape}')

    # ── Train R replicas for window B ──
    preds_B = np.zeros((R, len(idx_eval)), dtype=np.float32)
    per_rep_pr_auc_B = np.zeros(R, dtype=np.float32)
    if MODEL_TYPE == 'logreg':
        coef_B = np.zeros((R, X.shape[1]), dtype=np.float32)

    for r in range(R):
        # Offset by 5000 so A and B never share a bootstrap or model seed.
        boot_seed  = SEED_BASE + pid * 10_000 + 5_000 + r * 2
        model_seed = SEED_BASE + pid * 10_000 + 5_000 + r * 2 + 1
        boot_idx   = stratified_bootstrap(idx_B_boot, Y, seed=boot_seed)
        X_tr = X[boot_idx]
        Y_tr = Y[boot_idx]

        if MODEL_TYPE == 'xgboost':
            model, log = train_replica_with_es(X_tr, Y_tr, X_va_B, Y_va_B, hparams_B, model_seed)
            preds_B[r] = predict_proba_pos(model, X_eval)
            log.to_csv(dir_B / f'training_log_r{r}.csv', index=False)
        else:
            model = train_replica_logreg(X_tr, Y_tr, hparams_B, model_seed)
            preds_B[r] = model.predict_proba(X_eval)[:, 1]
            coef_B[r]  = model.named_steps['model'].coef_.ravel()

        joblib.dump(model, dir_B / f'model_r{r}.joblib', compress=3)
        per_rep_pr_auc_B[r] = average_precision_score(Y_eval, preds_B[r])
        with open(dir_B / f'seeds_r{r}.json', 'w') as f:
            json.dump({'bootstrap_seed': int(boot_seed),
                       'model_seed':     int(model_seed)}, f, indent=2)
        print(f'  B replica {r}: PR-AUC = {per_rep_pr_auc_B[r]:.4f}')

    if MODEL_TYPE == 'logreg':
        np.save(pair_dir / 'coef_B.npy', coef_B)
        print(f'  Saved coef_B.npy  shape={coef_B.shape}')

        # Full-window LR coefficient on window B (deterministic, no bootstrap).
        full_pipe_B = Pipeline([
            ('scaler', StandardScaler()),
            ('model',  LogisticRegression(**{**LR_FIXED, **hparams_B,
                                             'random_state': SEED_BASE + pid * 10 + 1})),
        ])
        full_pipe_B.fit(X[idx_B], Y[idx_B])
        coef_B_full = full_pipe_B.named_steps['model'].coef_.ravel().astype(np.float32)
        np.save(pair_dir / 'coef_B_full.npy', coef_B_full)
        print(f'  Saved coef_B_full.npy  shape={coef_B_full.shape}')
    # ── Aggregate, flag, score ──
    p_hat_A = preds_A.mean(axis=0)
    p_hat_B = preds_B.mean(axis=0)
    flagged_local_idx = compute_flagged_topk(p_hat_A, p_hat_B, K_FRAC)

    pr_auc_A  = average_precision_score(Y_eval, p_hat_A)
    pr_auc_B  = average_precision_score(Y_eval, p_hat_B)
    roc_auc_A = roc_auc_score(Y_eval, p_hat_A)
    roc_auc_B = roc_auc_score(Y_eval, p_hat_B)

    print(f'  Averaged: PR-AUC A = {pr_auc_A:.4f}, PR-AUC B = {pr_auc_B:.4f}')
    print(f'            ROC-AUC A = {roc_auc_A:.4f}, ROC-AUC B = {roc_auc_B:.4f}')
    print(f'  Flagged (top {K_FRAC:.0%}): {flagged_local_idx.shape[0]:,} / {len(idx_eval):,}')

    np.savez_compressed(
        pred_file,
        preds_A              = preds_A,
        preds_B              = preds_B,
        p_hat_A              = p_hat_A,
        p_hat_B              = p_hat_B,
        flagged_idx          = flagged_local_idx,
        Y_eval               = Y_eval,
        pr_auc_A             = np.float32(pr_auc_A),
        pr_auc_B             = np.float32(pr_auc_B),
        roc_auc_A            = np.float32(roc_auc_A),
        roc_auc_B            = np.float32(roc_auc_B),
        per_replica_pr_auc_A = per_rep_pr_auc_A,
        per_replica_pr_auc_B = per_rep_pr_auc_B,
    )

    performance_log.append({
        'pair_id':   pid,
        'pr_auc_A':  pr_auc_A,
        'pr_auc_B':  pr_auc_B,
        'roc_auc_A': roc_auc_A,
        'roc_auc_B': roc_auc_B,
        'n_flagged': flagged_local_idx.shape[0],
    })

print('\n✓ All window pairs processed.')


── Pair 00: A_end=2013-08  B_end=2013-10  eval=2013-11→2014-01  |A|=71,989 |B|=78,443 |eval|=23,346  PAIR_STRIDE=4 ──
  Tuning window A ...
    best A params: {'C': 0.061596216435321705}
  Tuning window B ...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_glm/_newton_solver.py:576: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
Ill-conditioned matrix (rcond=2.62922e-08): result may not be accurate.
  warnings.warn(


    best B params: {'C': 0.0432803176388536}
  A replica 0: PR-AUC = 0.8547
  A replica 1: PR-AUC = 0.8555
  A replica 2: PR-AUC = 0.8532
  Saved coef_A.npy  shape=(3, 317)
  Saved coef_A_full.npy  shape=(317,)
  B replica 0: PR-AUC = 0.8566
  B replica 1: PR-AUC = 0.8562
  B replica 2: PR-AUC = 0.8567
  Saved coef_B.npy  shape=(3, 317)
  Saved coef_B_full.npy  shape=(317,)
  Averaged: PR-AUC A = 0.8562, PR-AUC B = 0.8579
            ROC-AUC A = 0.9470, ROC-AUC B = 0.9469
  Flagged (top 10%): 2,335 / 23,346

── Pair 01: A_end=2013-12  B_end=2014-02  eval=2014-03→2014-05  |A|=74,193 |B|=76,161 |eval|=30,717  PAIR_STRIDE=4 ──
  Tuning window A ...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_glm/_newton_solver.py:576: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
Ill-conditioned matrix (rcond=2.19172e-08): result may not be accurate.
  warnings.warn(


    best A params: {'C': 0.06599356801746518}
  Tuning window B ...


...

    best B params: {'C': 0.04714427836471676}
  A replica 0: PR-AUC = 0.8722
  A replica 1: PR-AUC = 0.8736
  A replica 2: PR-AUC = 0.8726
  Saved coef_A.npy  shape=(3, 317)
  Saved coef_A_full.npy  shape=(317,)
  B replica 0: PR-AUC = 0.8731
  B replica 1: PR-AUC = 0.8733
  B replica 2: PR-AUC = 0.8729
  Saved coef_B.npy  shape=(3, 317)
  Saved coef_B_full.npy  shape=(317,)
  Averaged: PR-AUC A = 0.8745, PR-AUC B = 0.8745
            ROC-AUC A = 0.9555, ROC-AUC B = 0.9555
  Flagged (top 10%): 3,072 / 30,717

── Pair 02: A_end=2014-04  B_end=2014-06  eval=2014-07→2014-09  |A|=75,241 |B|=74,063 |eval|=25,883  PAIR_STRIDE=4 ──
  Tuning window A ...
    best A params: {'C': 0.038458432115618446}
  Tuning window B ...
    best B params: {'C': 0.06730995521220086}
  A replica 0: PR-AUC = 0.8454
  A replica 1: PR-AUC = 0.8458
  A replica 2: PR-AUC = 0.8457
  Saved coef_A.npy  shape=(3, 317)
  Saved coef_A_full.npy  shape=(317,)
  B replica 0: PR-AUC = 0.8461
  B replica 1: PR-AUC = 0.8456
  

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_glm/_newton_solver.py:576: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
Ill-conditioned matrix (rcond=2.32523e-08): result may not be accurate.
  warnings.warn(


    best A params: {'C': 0.035232565419238714}
  Tuning window B ...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_glm/_newton_solver.py:576: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
Ill-conditioned matrix (rcond=2.65941e-08): result may not be accurate.
  warnings.warn(


    best B params: {'C': 0.054125552363200075}
  A replica 0: PR-AUC = 0.8139
  A replica 1: PR-AUC = 0.8015
  A replica 2: PR-AUC = 0.8070
  Saved coef_A.npy  shape=(3, 317)
  Saved coef_A_full.npy  shape=(317,)
  B replica 0: PR-AUC = 0.8232
  B replica 1: PR-AUC = 0.8239
  B replica 2: PR-AUC = 0.8347
  Saved coef_B.npy  shape=(3, 317)
  Saved coef_B_full.npy  shape=(317,)
  Averaged: PR-AUC A = 0.8086, PR-AUC B = 0.8283
            ROC-AUC A = 0.9252, ROC-AUC B = 0.9327
  Flagged (top 10%): 2,854 / 28,536

✓ All window pairs processed.


## 4. Save and summarize performance

In [20]:
perf_df = pd.DataFrame(performance_log)
perf_df.to_csv(MODEL_DIR / 'performance_log.csv', index=False)

print(f'Saved {MODEL_DIR / "performance_log.csv"}')
print()
print(perf_df.round(4).to_string(index=False))

Saved /content/drive/MyDrive/Homesite_workspace/data/models/logreg/performance_log.csv

 pair_id  pr_auc_A  pr_auc_B  roc_auc_A  roc_auc_B  n_flagged
       0    0.8562    0.8579     0.9470     0.9469       2335
       1    0.8745    0.8745     0.9555     0.9555       3072
       2    0.8472    0.8472     0.9472     0.9468       2589
       3    0.7980    0.8086     0.9184     0.9268       2414
       4    0.8086    0.8283     0.9252     0.9327       2854
